In [ ]:
# IST 707 - Environment Setup
# Run this cell first to install required packages.
# On Colab: if packages need upgrading, the runtime restarts automatically.
#           Just re-run this cell after the restart.
import subprocess, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import numpy as _np
    _old_np = _np.__version__

    !pip install -q -r https://raw.githubusercontent.com/pjmcswee/IST707-Notebooks/main/requirements.txt

    import importlib.metadata
    _new_np = importlib.metadata.version('numpy')

    if _old_np != _new_np:
        print(f'\n\u26a0\ufe0f  numpy upgraded ({_old_np} \u2192 {_new_np}). Restarting runtime...')
        print('\u27a1\ufe0f  Re-run this cell after restart (it will be instant).')
        import os
        os.kill(os.getpid(), 9)
    else:
        print(f'\n\u2705 Setup complete! Python: {sys.version.split()[0]}, numpy: {_old_np}')
else:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', '../requirements.txt']
    )
    print(f'\n\u2705 Setup complete! Python: {sys.version.split()[0]}')

# Week 1 In-Class Exercise: Is Education Linked to Income?

In the textbook (Chapter 1), we explored whether **GDP per capita predicts life satisfaction**.

In this exercise, you'll explore a similar question with a different dataset:

> **Does education level predict median household income across US states?**

You'll practice:
- Loading and exploring data (EDA)
- Visualizing relationships with scatter plots
- Fitting a **linear regression** model
- Making predictions
- Comparing to a **k-Nearest Neighbors** model
- Observing overfitting

These are the same concepts from the slides:
- Model-based vs. instance-based learning
- The ML workflow (define → explore → train → evaluate)
- Overfitting & the bias-variance tradeoff

## Part 1: Load and Explore the Data

We'll use a dataset of US states with:
- `pct_bachelors` — % of population with a bachelor's degree or higher
- `median_income` — median household income (USD)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# US Census-derived data: education vs income by state
# Source: American Community Survey (ACS) 2022 estimates
data = {
    'state': ['Mississippi', 'West Virginia', 'Arkansas', 'Louisiana', 'Alabama',
              'Kentucky', 'New Mexico', 'Oklahoma', 'Tennessee', 'South Carolina',
              'Indiana', 'Ohio', 'Missouri', 'Michigan', 'North Carolina',
              'Georgia', 'Texas', 'Florida', 'Pennsylvania', 'Illinois',
              'Minnesota', 'Virginia', 'Washington', 'Colorado', 'Massachusetts',
              'New Jersey', 'Connecticut', 'California', 'New Hampshire', 'Maryland'],
    'pct_bachelors': [23.2, 22.7, 24.3, 25.0, 27.1,
                      25.8, 28.3, 26.9, 29.0, 29.4,
                      28.2, 30.1, 30.5, 31.2, 33.4,
                      33.7, 32.2, 33.3, 35.7, 36.5,
                      38.3, 40.8, 38.0, 42.7, 45.2,
                      42.1, 40.4, 35.3, 38.4, 42.0],
    'median_income': [52985, 55515, 56335, 57852, 59609,
                      60183, 58722, 61364, 63426, 63623,
                      64321, 65720, 65920, 66986, 67784,
                      71355, 73035, 67917, 73170, 78433,
                      84223, 87249, 90325, 87598, 96505,
                      97126, 90213, 91905, 90845, 98461]
}

df = pd.DataFrame(data)
df = df.set_index('state')
df = df.sort_values('pct_bachelors')
print(f"Dataset: {len(df)} US states")
df.head(10)

### Exercise 1.1: Basic EDA

Run the cell below to see summary statistics. Then answer:
- What's the range of education levels?
- What's the range of incomes?
- Do you think there's a relationship?

In [ ]:
df.describe()

## Part 2: Visualize the Data

Just like the GDP vs. Life Satisfaction example in the slides, let's make a scatter plot.

In [ ]:
df.plot(kind='scatter', x='pct_bachelors', y='median_income',
        figsize=(8, 5), grid=True)
plt.xlabel("% with Bachelor's Degree or Higher")
plt.ylabel("Median Household Income (USD)")
plt.title("Education vs. Income Across US States")
plt.tight_layout()
plt.show()

### Exercise 2.1: Interpret the Plot

**Question:** Does there appear to be a linear relationship? Is it perfect, or noisy?

*Write your answer here (double-click to edit):*

...

## Part 3: Model-Based Learning — Linear Regression

Let's fit a linear model, just like in the textbook example:

$$\text{income} = \theta_0 + \theta_1 \times \text{pct\_bachelors}$$

This is **model-based learning** — we're building a mathematical model from the data.

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare the data
X = df[['pct_bachelors']].values  # Features (2D array)
y = df[['median_income']].values  # Target

# Train the model
model_lr = LinearRegression()
model_lr.fit(X, y)

# Print the learned parameters
print(f"θ₀ (intercept): {model_lr.intercept_[0]:,.0f}")
print(f"θ₁ (slope):     {model_lr.coef_[0][0]:,.0f}")
print(f"\nInterpretation: Each additional 1% of bachelor's degree holders")
print(f"is associated with ~${model_lr.coef_[0][0]:,.0f} higher median income.")

In [ ]:
# Visualize the linear model
df.plot(kind='scatter', x='pct_bachelors', y='median_income',
        figsize=(8, 5), grid=True)

X_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
plt.plot(X_range, model_lr.predict(X_range), 'r-', linewidth=2,
         label='Linear Regression')

plt.xlabel("% with Bachelor's Degree or Higher")
plt.ylabel("Median Household Income (USD)")
plt.title("Linear Model: Education → Income")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 3.1: Make a Prediction

Use the model to predict the median income for a state where **35%** of the population has a bachelor's degree.

In [ ]:
# TODO: Replace the ??? with the correct value
new_state_education = [[???]]  # 35% bachelor's degree
predicted_income = model_lr.predict(new_state_education)
print(f"Predicted median income: ${predicted_income[0][0]:,.0f}")

## Part 4: Instance-Based Learning — k-Nearest Neighbors

Instead of building a model, what if we just looked at the **most similar states**?

This is **instance-based learning** — the system memorizes the training data and uses similarity to make predictions.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Train a KNN model (k=3)
model_knn = KNeighborsRegressor(n_neighbors=3)
model_knn.fit(X, y)

# Predict for the same state (35% bachelor's)
knn_prediction = model_knn.predict([[35]])
print(f"KNN prediction (k=3): ${knn_prediction[0][0]:,.0f}")
print(f"Linear prediction:    ${predicted_income[0][0]:,.0f}")

In [ ]:
# Visualize KNN vs Linear Regression
df.plot(kind='scatter', x='pct_bachelors', y='median_income',
        figsize=(8, 5), grid=True)

X_range = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
plt.plot(X_range, model_lr.predict(X_range), 'r-', linewidth=2,
         label='Linear Regression')
plt.plot(X_range, model_knn.predict(X_range), 'g--', linewidth=2,
         label='KNN (k=3)')

plt.xlabel("% with Bachelor's Degree or Higher")
plt.ylabel("Median Household Income (USD)")
plt.title("Model-Based (Linear) vs Instance-Based (KNN)")
plt.legend()
plt.tight_layout()
plt.show()

### Exercise 4.1: Try Different Values of k

**Question:** What happens when you change `n_neighbors` to 1? To 10? To 29 (almost all the data)?

Which values overfit? Which underfit?

In [ ]:
# TODO: Experiment with different values of k
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k in zip(axes, [1, 5, 29]):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X, y)
    
    ax.scatter(df['pct_bachelors'], df['median_income'], alpha=0.7)
    ax.plot(X_range, knn.predict(X_range), 'g-', linewidth=2)
    ax.set_title(f'KNN (k={k})')
    ax.set_xlabel("% Bachelor's")
    ax.set_ylabel("Income")
    ax.grid(True)

plt.tight_layout()
plt.show()

### Exercise 4.2: Bias-Variance Tradeoff

Look at the three plots above:

- **k=1**: Does the model have high bias or high variance? Is it overfitting or underfitting?
- **k=29**: Does the model have high bias or high variance? Is it overfitting or underfitting?
- **k=5**: Where does it fall on the tradeoff?

*Write your answers here:*

- k=1: ...
- k=29: ...
- k=5: ...

## Part 5: Train/Test Split

So far we've trained AND evaluated on the same data. That's cheating!

Let's split the data into training and test sets to get an honest evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} states")
print(f"Test set:     {len(X_test)} states")

In [ ]:
# Train both models on training data only
lr = LinearRegression().fit(X_train, y_train)
knn3 = KNeighborsRegressor(n_neighbors=3).fit(X_train, y_train)
knn1 = KNeighborsRegressor(n_neighbors=1).fit(X_train, y_train)

# Evaluate on test data
models = {'Linear Regression': lr, 'KNN (k=3)': knn3, 'KNN (k=1)': knn1}

print(f"{'Model':<20} {'Train RMSE':>12} {'Test RMSE':>12}")
print('-' * 46)
for name, m in models.items():
    train_rmse = root_mean_squared_error(y_train, m.predict(X_train))
    test_rmse = root_mean_squared_error(y_test, m.predict(X_test))
    print(f"{name:<20} ${train_rmse:>10,.0f} ${test_rmse:>10,.0f}")

### Exercise 5.1: Interpret the Results

**Questions:**
1. Which model has the lowest training error? Is that surprising?
2. Which model generalizes best (lowest test error)?
3. Which model is overfitting? How can you tell from the train vs. test gap?

*Write your answers here:*

1. ...
2. ...
3. ...

## Summary: Connecting to This Week's Concepts

| Concept from Slides | Where you saw it |
|---|---|
| Model-based learning | Linear Regression (Part 3) |
| Instance-based learning | KNN (Part 4) |
| Overfitting | KNN with k=1 |
| Underfitting | KNN with k=29 |
| Bias-variance tradeoff | Exercise 4.2 |
| Train/test split | Part 5 |
| The ML workflow | Load → Explore → Train → Evaluate |

### What's next?
Next week we'll do a **full end-to-end ML project** with a real-world dataset — including data cleaning, feature engineering, and multiple model comparison.